# Rate constant Calculation and Diagrams

In [1]:
import jax
import jax.numpy as jnp
from diffPLOG2TROE.kinetic_constants import Arrhenius, FallOff, Plog

## Modified Arrhenius
This tutorial demonstrates the computation of kinetic rate constants for a simple Arrhenius reaction across a specified temperature range. The implementation focuses solely on the forward reaction rate constant, as determined by user-provided parameters. Note that this code does not incorporate chemical equilibrium calculations, thus considering only unidirectional reaction kinetics.
- Equation
    > $k_f(T) = A \: T^b \: exp\left(\frac{Eact}{RT}\right)$
- CHEMKIN representation
    >```
    >H2 + O = H + OH    +5.080E+04 +2.670E+00 +6.292E+03
    >```
- Internal representation
    >```python
    >rate_constant = {
    >    "name": "H2+O=H+OH",
    >    "type": "arrhenius",
    >    "rate-constant": {
    >        "coefficients": [5.080e+04, 2.670E+00, 6.292e+03]
    >    }
    >}
    >```
    >As an alternative option, the "name" of the reaction can be specified as follows to facilitate subsequent handling during the plotting operations.
    >```python
    >rate_constant = {
    >    "name": "\\ce{H2 + O = H + OH}",
    >    "type": "arrhenius",
    >    "rate-constant": {
    >        "coefficients": [5.080e+04, 2.670E+00, 6.292e+03]
    >    }
    >}
    >```

In [2]:
# --------------------------------------------------------
# Dictionary initialization
# --------------------------------------------------------
rate_constant = {
    "name": "H2+O=H+OH",
    "type": "arrhenius",
    "rate-constant": {
        "coefficients": [5.080e+04, 2.670E+00, 6.292e+03]
    }
}

constant = Arrhenius(rate_constant)
print(constant)

# --------------------------------------------------------
# Direct parameters initialization
# --------------------------------------------------------
constant_named = Arrhenius(name="H2+O=H+OH", params=jnp.array([5.080e+04, 2.670E+00, 6.292e+03]))
constant_unnamed = Arrhenius(params=jnp.array([5.080e+04, 2.670E+00, 6.292e+03]))  # Reaction name can be omitted
print(constant_named)
print(constant_unnamed)

H2+O=H+OH		5.08000e+04 2.67000e+00 6.29200e+03
H2+O=H+OH		5.08000e+04 2.67000e+00 6.29200e+03
unknown :(		5.08000e+04 2.67000e+00 6.29200e+03


## Fall-Off Reactions
- Equations
    > $k_f \: (T, P_r) = k_{\infty} \left(\dfrac{P_{r}}{1+P_{r}}\right)F(T, P_r) $
    >
    > $P_r$ is the reduce pressure computed as follow:
    > $P_r = \dfrac{k_{0}[M]}{k_{\infty}}$
    > 
    > $[M]$ is the concentration of the mixture, possibly including enhanced third-body efficiencies.
    >
    > $k_0$ and $k_{\infty}$ are the low pressure and high pressure limits of the rate constant computed as a classic Arrhenius constant (see above).
    >
    > $F(T, P_r)$ is the falloff function that depending on different formalism can assume different values:
    > | | |
    > |:- |:- |
    > | **Lindemann** | $F = 1$|
    > | **Troe**      | $log_{10}F = \dfrac{log_{10}Fcent}{1 + f_{1}^2}$<br/><br/>$Fcent = (1-A) exp\left(-\dfrac{T}{T_3}\right) + Aexp\left(-\dfrac{T}{T_1}\right) + exp\left(\dfrac{T_2}{T}\right)$<br/><br/>$f_1 = \dfrac{log_{10}P_{r} + c}{n - 0.14\left(log_{10}P_{r} + c\right)}$<br/><br/>$c = -0.4 - 0.67log_{10}Fcent$<br/><br/>$n = 0.75 - 1.27 log_{10}Fcent$ |
    > | **SRI**       | $X = \dfrac{1}{1 + log_{10}^{2}P_{r}}$<br/><br/>$F = d\left[a\times exp\left(-b/T\right) + exp\left(-T/c\right)\right]^{X} T^{e}$ |


- CHEMKIN representation (Examples are courtesy of the CHEMKIN manual)
    - **Lindemann**
      >```
      > H + C2H4(+M) = C2H5(+M)    0.221E+14  0.000  2066.0   ! Michael
      >  LOW /                     6.369E+27 -2.760 -54.000 / ! Lindemann fall-off reaction
      > H2/ 2/ CO/ 2/ CO2/ 3/ H2O/ 5/                         ! enhanced third-body efficiencies
      >```
    - **TROE**
      >```
      > CH3 + CH3(+M) = C2H6(+M)   9.030E+16 -1.180 654.000 
      >  LOW /                     3.180E+41 -7.030 2762.00 /
      > TROE / 0.6041 6927.00 132.00 0.000                  / ! TROE fall-off reaction, with 4 parameters the fourth one is optional
      > H2/ 2/ CO/ 2/ CO2/ 3/ H2O/ 5/                         ! enhanced third-body efficiencies
      >```
    - **SRI**
      >```
      > CH3 + H(+M) = CH4(+M)      6.000E+16 -1.000 0.000 
      >  LOW /                     8.000E+26 -3.000 0.000 /
      > SRI  / 0.450 797.00 979.00 0.000 0.000            / ! SRI fall-off reaction
      > H2/ 2/ CO/ 2/ CO2/ 3/ H2O/ 5/                       ! enhanced third-body efficiencies
      >```

- Internal representation
    >```python
    >lindemann_constant = {
    >    "name": "H+C2H4(+M)=C2H5(+M)",
    >    "type": "falloff",
    >    "falloff-type": "lindemann",
    >    "rate-constant": {
    >        "lpl-coefficients": [6.369e+27, -2.760, -54.000],
    >        "hpl-coefficients": [0.221e+14, 0.000, 2066.0],
    >    }
    >}
    >
    >troe_constant = {
    >    "name": "CH3+CH3(+M)=C2H6(+M)",
    >    "type": "falloff",
    >    "falloff-type": "troe",
    >    "rate-constant": {
    >        "lpl-coefficients": [9.030e+16, -1.180, 654.000],
    >        "hpl-coefficients": [6.369e+27, -2.760, -54.000],
    >        "falloff-coefficients": [0.6041, 6927.00, 132.00],
    >        "efficiencies": {"H2": 2, "CO": 2, "CO2": 3, "H2O": 5}
    >    }
    >}
    >
    >sri_constant = {
    >    "name": "CH3+H(+M)=CH4(+M)",
    >    "type": "falloff",
    >    "falloff-type": "sri",
    >    "rate-constant": {
    >        "lpl-coefficients": [6.000e+16, -1.000, 0.000],
    >        "hpl-coefficients": [8.000e+26, -3.000, 0.000],
    >        "falloff-coefficients": [0.450, 797.00, 979.00],
    >        "efficiencies": {"H2": 2, "CO": 2, "CO2": 3, "H2O": 5}
    >    }
    >}
```

In [3]:
# --------------------------------------------------------
# Dictionary initialization
# --------------------------------------------------------
lindemann_constant = {
    "name": "H+C2H4(+M)=C2H5(+M)",
    "type": "falloff",
    "falloff-type": "lindemann",
    "rate-constant": {
        "lpl-coefficients": [6.369e+27, -2.760, -54.000],
        "hpl-coefficients": [0.221e+14, 0.000, 2066.0],
    }
}
constant = FallOff(lindemann_constant)
print(constant)

troe_constant = {
    "name": "CH3+CH3(+M)=C2H6(+M)",
    "type": "falloff",
    "falloff-type": "troe",
    "rate-constant": {
        "lpl-coefficients": [9.030E+16, -1.180, 654.000],
        "hpl-coefficients": [6.369E+27, -2.760, -54.000],
        "falloff-coefficients": [0.6041, 6927.00, 132.00, 0.0],
        "efficiencies": {"H2": 2, "CO": 2, "CO2": 3, "H2O": 5}
    }
}
constant = FallOff(troe_constant)
print(constant)

sri_constant = {
    "name": "CH3+H(+M)=CH4(+M)",
    "type": "falloff",
    "falloff-type": "sri",
    "rate-constant": {
        "lpl-coefficients": [6.000e+16, -1.000, 0.000],
        "hpl-coefficients": [8.000e+26, -3.000, 0.000],
        "falloff-coefficients": [0.450, 797.00, 979.00],
        "efficiencies": {"H2": 2, "CO": 2, "CO2": 3, "H2O": 5}
    }
}
constant = FallOff(sri_constant)
print(constant)

# --------------------------------------------------------
# Direct parameters initialization
# --------------------------------------------------------
print("\nParameter initialization\n")
constant = FallOff(
    name="H+C2H4(+M)=C2H5(+M)",
    lpl_params=jnp.array([6.369e+27, -2.760, -54.000]),
    hpl_params=jnp.array([8.000e+26, -3.000, 0.000]),
    falloff_type="lindemann"
)
print(constant)

constant = FallOff(
    name="CH3+CH3(+M)=C2H6(+M)",
    lpl_params=jnp.array([9.030E+16, -1.180, 654.000]),
    hpl_params=jnp.array([6.369E+27, -2.760, -54.000]),
    falloff_params=jnp.array([0.6041, 6927.00, 132.00, 0.0]),
    falloff_type="troe"
)
print(constant)

H+C2H4(+M)=C2H5(+M)		2.21000e+13 0.00000 2.06600e+03
 LOW / 		6.36900e+27 -2.76000 -5.40000e+01 /


CH3+CH3(+M)=C2H6(+M)		6.36900e+27 -2.76000 -5.40000e+01
 LOW / 		9.03000e+16 -1.18000 6.54000e+02 /
 TROE / 6.04100e-01 6.92700e+03 1.32000e+02 0.00000e+00 /

CH3+H(+M)=CH4(+M)		8.00000e+26 -3.00000 0.00000e+00
 LOW / 		6.00000e+16 -1.00000 0.00000e+00 /
 SRI / 4.50000e-01 7.97000e+02 9.79000e+02 0.00000e+00 0.00000e+00 /


Parameter initialization

H+C2H4(+M)=C2H5(+M)		8.00000e+26 -3.00000 0.00000e+00
 LOW / 		6.36900e+27 -2.76000 -5.40000e+01 /


CH3+CH3(+M)=C2H6(+M)		6.36900e+27 -2.76000 -5.40000e+01
 LOW / 		9.03000e+16 -1.18000 6.54000e+02 /
 TROE / 6.04100e-01 6.92700e+03 1.32000e+02 0.00000e+00 /



## Pressure Logarithmic Reaction
- Equation
- CHEMKIN representation
- Internal representation
    >```python
    >plog_constant = {
    >    "name": "",
    >    "type": "plog",
    >    "rate-constant": {
    >        "coefficients": [
    >            [0.01, 6.369e+27, -2.760, -54.000],
    >            [0.01, 6.369e+27, -2.760, -54.000],
    >            [0.01, 6.369e+27, -2.760, -54.000],
    >            [0.01, 6.369e+27, -2.760, -54.000],
    >            [0.01, 6.369e+27, -2.760, -54.000],
    >         ]
    >    }
    >}
    >```

In [4]:
# --------------------------------------------------------
# Dictionary initialization
# --------------------------------------------------------
plog_constant = {
    "name": "NH3+H=NH2+H2",
    "type": "plog",
    "rate-constant": {
        "coefficients": [
            [1.000000e-01, 7.230000e29, -5.316000e00, 1.108624e05],
            [1.000000e00, 3.497000e30, -5.224000e00, 1.111633e05],
            [1.000000e01, 1.975000e31, -5.160000e00, 1.118878e05],
            [1.000000e02, 2.689000e31, -4.920000e00, 1.127787e05],
        ]
    }
}

constant = Plog(plog_constant)
print(constant)

# --------------------------------------------------------
# Direct parameter initialization
# --------------------------------------------------------
constant = Plog(
    # name="NH3+H=NH2+H2", Can be omitted
    params=jnp.array([
        [1.000000e-01, 7.230000e29, -5.316000e00, 1.108624e05],
        [1.000000e00, 3.497000e30, -5.224000e00, 1.111633e05],
        [1.000000e01, 1.975000e31, -5.160000e00, 1.118878e05],
        [1.000000e02, 2.689000e31, -4.920000e00, 1.127787e05],
    ])
)
print(constant)

NH3+H=NH2+H2		0.00000e+00 0.00000 0.00000e+00
 PLOG / 1.00000e-01	7.23000e+29 -5.31600 1.10862e+05 /
 PLOG / 1.00000e+00	3.49700e+30 -5.22400 1.11163e+05 /
 PLOG / 1.00000e+01	1.97500e+31 -5.16000 1.11888e+05 /
 PLOG / 1.00000e+02	2.68900e+31 -4.92000 1.12779e+05 /

unknown :(		0.00000e+00 0.00000 0.00000e+00
 PLOG / 1.00000e-01	7.23000e+29 -5.31600 1.10862e+05 /
 PLOG / 1.00000e+00	3.49700e+30 -5.22400 1.11163e+05 /
 PLOG / 1.00000e+01	1.97500e+31 -5.16000 1.11888e+05 /
 PLOG / 1.00000e+02	2.68900e+31 -4.92000 1.12779e+05 /

